In [1]:
# Verifica as GPUs que estão sendo utilizadas
!nvidia-smi

Tue Nov 17 23:14:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.99.02              Driver Version: 595.99.02      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5070 Ti     Off |   00000000:01:00.0  On |                  N/A |
|  0%   49C    P1             52W /  300W |    1195MiB /  16303MiB |      5%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="0" #aqui tem q escolher uma das gpus, veja a que esta desocupada

In [3]:
tf_device='/gpu:0'

In [4]:
import os
import warnings

# Bloqueia logs de C++ (INFO e WARNING)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 

# CORREÇÃO: Em vez de /dev/null, desabilite o salvamento de MLIR completamente
os.environ['TF_MLIR_ENABLE_MLIR_HAL_TARGET_ EXECUTABLE_DUMP_PATH'] = ''
os.environ['MLIR_CRASH_REPRODUCER_DIRECTORY'] = ''

import tensorflow as tf
import logging

# Silencia os loggers do Python
logging.getLogger('tensorflow').setLevel(logging.ERROR)
logging.getLogger('absl').setLevel(logging.ERROR)

# Garante que o XLA não polua o console
tf.config.optimizer.set_jit(True)

I0000 00:00:1794968100.269442  610918 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


# Exemplo: https://keras.io/examples/vision/mnist_convnet/

# Setup

In [5]:
!pip install watermark

In [6]:
import numpy as np
import keras
from keras import layers

# Versões dos pacotes usados neste jupyter notebook
%reload_ext watermark
%watermark -a "Clésio Gonçalves" --iversions
!python --version

Author: Clésio Gonçalves

keras     : 3.14.1
logging   : 0.5.1.2
numpy     : 2.3.5
tensorflow: 2.21.0

Python 3.13.9


# Prepare the data

In [7]:
# Model / data parameters
num_classes = 10
input_shape = (28, 28, 1)

# Load the data and split it between train and test sets
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Scale images to the [0, 1] range
x_train = x_train.astype("float32") / 255
x_test = x_test.astype("float32") / 255
# Make sure images have shape (28, 28, 1)
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)
print("x_train shape:", x_train.shape)
print(x_train.shape[0], "train samples")
print(x_test.shape[0], "test samples")


# convert class vectors to binary class matrices
y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)


x_train shape: (60000, 28, 28, 1)
60000 train samples
10000 test samples


# Build the model



In [8]:
model = keras.Sequential(
    [
        keras.Input(shape=input_shape),
        layers.Conv2D(32, kernel_size=(3, 3), activation="relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Conv2D(64, kernel_size=(3, 3), activation="relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Flatten(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax"),
    ]
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        16,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 34,826 (136.04 KB)

 Trainable params: 34,826 (136.04 KB)

 Non-trainable params: 0 (0.00 B)

# Train the model

In [9]:
batch_size = 128
epochs = 15

model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_split=0.1)

Epoch 1/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.8898 - loss: 0.3666 - val_accuracy: 0.9758 - val_loss: 0.0852
Epoch 2/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 911us/step - accuracy: 0.9649 - loss: 0.1131 - val_accuracy: 0.9852 - val_loss: 0.0578
Epoch 3/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 928us/step - accuracy: 0.9742 - loss: 0.0851 - val_accuracy: 0.9865 - val_loss: 0.0499
Epoch 4/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 920us/step - accuracy: 0.9776 - loss: 0.0719 - val_accuracy: 0.9900 - val_loss: 0.0419
Epoch 5/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 933us/step - accuracy: 0.9802 - loss: 0.0618 - val_accuracy: 0.9890 - val_loss: 0.0414
Epoch 6/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 922us/step - accuracy: 0.9827 - loss: 0.0567 - val_accuracy: 0.9902 - val_loss: 0.0373
Epoch 7/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 940us/step - accuracy: 0.9839 - loss: 0.0523 - val_accuracy: 0.9903 - val_loss: 0.0328
Epoch 8/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 923us/step - accuracy: 0.9848 - loss: 0.0482 - va

# Evaluate the trained model

In [10]:
score = model.evaluate(x_test, y_test, verbose=0)
print("Test loss:", score[0])
print("Test accuracy:", score[1])


Test loss: 0.02278761751949787
Test accuracy: 0.9925000071525574
